In [1]:
from kaggle_secrets import UserSecretsClient
secret_label = "HF_TOKEN"
secret_value = UserSecretsClient().get_secret(secret_label)

In [2]:
import os
os.environ["HF_TOKEN"] = secret_value

In [3]:
import random
import numpy as np
import tensorflow as tf
from glob import glob
from sklearn.model_selection import train_test_split

2026-01-28 13:35:25.519990: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769607325.715604      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769607325.768370      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769607326.227001      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769607326.227034      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769607326.227037      55 computation_placer.cc:177] computation placer alr

In [4]:
from transformers import TFWav2Vec2Model, Wav2Vec2Config
import librosa

In [5]:
class Config:
    # XLSR-53: The 300M param beast for multilingual speech
    MODEL_NAME = "facebook/wav2vec2-large-xlsr-53"
    
    SR = 16000
    MAX_DURATION = 5.0
    INPUT_LEN = int(SR * MAX_DURATION)
    
    # Inference
    DEPLOY_THRESHOLD = 0.73  # Needs calibration on your validation set
    SLIDING_WINDOW_STRIDE = 2.0  # Seconds
    
    # Training
    BATCH_SIZE = 4
    LEARNING_RATE = 1e-4  # Standard LR is safe because we freeze the backbone

config = Config()

DATASET_DIR = "/kaggle/input/ironwall-ai-audio-data/dataset"
REAL_DIR = os.path.join(DATASET_DIR, "real")
FAKE_DIR = os.path.join(DATASET_DIR, "fake")

EPOCHS = 2
HOURS_PER_EPOCH = 300
CLIP_SECONDS = 5.0

TRAIN_SPLIT = 0.8
VAL_SPLIT = 0.1
TEST_SPLIT = 0.1

CLASS_WEIGHT = {0: 1.0, 1: 2.0}

CHECKPOINT_DIR = "checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

In [6]:
tf.keras.mixed_precision.set_global_policy("float32")

strategy = tf.distribute.MirroredStrategy()
NUM_GPUS = strategy.num_replicas_in_sync
print(f"Using {NUM_GPUS} GPUs")

INFO:tensorflow:Using MirroredStrategy with devices ('/job:localhost/replica:0/task:0/device:GPU:0', '/job:localhost/replica:0/task:0/device:GPU:1')
Using 2 GPUs


I0000 00:00:1769607352.926664      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13942 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1769607352.927389      55 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13942 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


In [7]:
def process_audio_chunk(y, target_len=config.INPUT_LEN):
    """
    Core logic to normalize and mask a raw audio array.
    Used by both training (file loader) and inference (sliding window).
    """
    # --- FIX 1 (Normalization) ---
    # Safe normalization to [-1, 1] range
    peak = np.max(np.abs(y))
    if peak > 0:
        y = y / (peak + 1e-9)
        
    # Pad or Crop
    if len(y) > target_len:
        # Center crop
        start = (len(y) - target_len) // 2
        y = y[start : start + target_len]
        mask = np.ones(target_len, dtype=np.int32)
    else:
        # --- FIX 2 (Attention Masking) ---
        pad_len = target_len - len(y)
        mask = np.concatenate([np.ones(len(y)), np.zeros(pad_len)])
        y = np.pad(y, (0, pad_len), mode='constant')
        
    return y.astype(np.float32), mask.astype(np.int32)


def preprocess_from_file(path):
    """Wrapper to load from file"""
    try:
        y, _ = librosa.load(path, sr=config.SR, mono=True)
        y, _ = librosa.effects.trim(y, top_db=20)
        return process_audio_chunk(y)
    except Exception as e:
        print(f"Error processing {path}: {e}")
        return np.zeros(config.INPUT_LEN), np.zeros(config.INPUT_LEN)


In [8]:
def build_secure_model():
    # Load Config
    hf_config = Wav2Vec2Config.from_pretrained(config.MODEL_NAME)
    hf_config.output_hidden_states = True
    
    # Load Model
    wav2vec2 = TFWav2Vec2Model.from_pretrained(config.MODEL_NAME, config=hf_config, from_pt=True)
    
     # 🔒 Freeze EVERYTHING first
    wav2vec2.trainable = False

    # 🔓 Unfreeze top N transformer layers
    encoder_layers = wav2vec2.wav2vec2.encoder.layer
    total_layers = len(encoder_layers)

    UNFREEZE_LAST_N = 6  # ~top 25%

    for layer in encoder_layers[-UNFREEZE_LAST_N:]:
        layer.trainable = True

    print(f"Total transformer layers found: {total_layers}")
    print(f"Unfrozen top layers: {UNFREEZE_LAST_N}")

        
    # Inputs
    input_audio = tf.keras.layers.Input(shape=(config.INPUT_LEN,), dtype=tf.float32, name="audio_in")
    input_mask  = tf.keras.layers.Input(shape=(config.INPUT_LEN,), dtype=tf.int32, name="mask_in")
    
    # Forward Pass with Mask
    x = wav2vec2(input_audio, attention_mask=input_mask).last_hidden_state
    
    # --- FIX 3: DYNAMIC ATTENTION POOLING ---
    # Calculate attention scores
    # Shape: (Batch, Time, 1)
    att_scores = tf.keras.layers.Dense(1, activation='tanh')(x) 
    
    # Softmax over the Time dimension (axis 1)
    att_weights = tf.keras.layers.Softmax(axis=1)(att_scores)
    
    # Apply weights: x * weights
    # Broadcasting handles the dimensions: (Batch, Time, 1024) * (Batch, Time, 1)
    x = tf.keras.layers.Multiply()([x, att_weights])
    
    # Sum over time to get context vector
    x = tf.keras.layers.Lambda(lambda t: tf.reduce_sum(t, axis=1))(x)
    
    # Classification Head
    x = tf.keras.layers.Dense(256, activation="relu")(x)
    x = tf.keras.layers.Dropout(0.5)(x)
    output = tf.keras.layers.Dense(1, activation="sigmoid")(x)
    
    model = tf.keras.Model(inputs=[input_audio, input_mask], outputs=output)
    
    # --- FIX 2: UNIFIED OPTIMIZER ---
    # Since we froze the deep layers, a standard LR is safe.
    opt = tf.keras.optimizers.Adam(learning_rate=config.LEARNING_RATE)
    
    model.compile(
        loss="binary_crossentropy", # Use class_weight in fit() for penalty
        optimizer=opt, 
        metrics=["accuracy"]
    )
    return model


In [9]:
def load_file_list():
    real_files = glob(os.path.join(REAL_DIR, "*.wav"))
    fake_files = glob(os.path.join(FAKE_DIR, "*.wav"))

    print(f"Found {len(real_files)} real files")
    print(f"Found {len(fake_files)} fake files")

    return real_files, fake_files


def split_dataset(real_files, fake_files):
    # Split REAL
    real_train, real_tmp = train_test_split(
        real_files, train_size=TRAIN_SPLIT, random_state=SEED
    )
    real_val, real_test = train_test_split(
        real_tmp,
        test_size=TEST_SPLIT / (VAL_SPLIT + TEST_SPLIT),
        random_state=SEED
    )

    # Split FAKE
    fake_train, fake_tmp = train_test_split(
        fake_files, train_size=TRAIN_SPLIT, random_state=SEED
    )
    fake_val, fake_test = train_test_split(
        fake_tmp,
        test_size=TEST_SPLIT / (VAL_SPLIT + TEST_SPLIT),
        random_state=SEED
    )

    return (
        (real_train, fake_train),
        (real_val, fake_val),
        (real_test, fake_test),
    )


def sample_subset(real_files, fake_files, hours_target):
    clips_needed = int(hours_target * 3600 / CLIP_SECONDS)
    half = clips_needed // 2

    real_sample = random.sample(real_files, min(half, len(real_files)))
    fake_sample = random.sample(fake_files, min(half, len(fake_files)))

    files = real_sample + fake_sample
    labels = [0] * len(real_sample) + [1] * len(fake_sample)

    combined = list(zip(files, labels))
    random.shuffle(combined)

    files, labels = zip(*combined)
    return list(files), list(labels)



In [10]:
class AudioGenerator(tf.keras.utils.Sequence):
    def __init__(self, files, labels, batch_size, training=True):
        self.files = files
        self.labels = labels
        self.batch_size = batch_size
        self.training = training
        self.indices = np.arange(len(files))
        self.on_epoch_end()

    def __len__(self):
        return int(np.ceil(len(self.files) / self.batch_size))

    def __getitem__(self, idx):
        batch_idx = self.indices[idx * self.batch_size:(idx + 1) * self.batch_size]

        X_audio, X_mask, y = [], [], []

        for i in batch_idx:
            audio, mask = preprocess_from_file(self.files[i])

            # light augmentation (train only)
            if self.training and random.random() < 0.3:
                t = random.randint(0, len(audio) - 400)
                audio[t:t + 400] = 0

            X_audio.append(audio)
            X_mask.append(mask)
            y.append(self.labels[i])

        return (
            {"audio_in": np.array(X_audio), "mask_in": np.array(X_mask)},
            np.array(y),
        )

    def on_epoch_end(self):
        if self.training:
            np.random.shuffle(self.indices)



In [11]:
print("GPUs:", tf.config.list_physical_devices("GPU"))

GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]


In [12]:
real_files, fake_files = load_file_list()

(real_train, fake_train), (real_val, fake_val), (real_test, fake_test) = split_dataset(
    real_files, fake_files
)

print(f"Train: {len(real_train)} real / {len(fake_train)} fake")
print(f"Val:   {len(real_val)} real / {len(fake_val)} fake")
print(f"Test:  {len(real_test)} real / {len(fake_test)} fake")

Found 227551 real files
Found 399206 fake files
Train: 182040 real / 319364 fake
Val:   22755 real / 39921 fake
Test:  22756 real / 39921 fake


In [13]:
val_files = real_val + fake_val
val_labels = [0] * len(real_val) + [1] * len(fake_val)

val_gen = AudioGenerator(
    val_files, val_labels,
    batch_size=config.BATCH_SIZE,
    training=False
)

In [14]:
with strategy.scope():
    model = build_secure_model()

config.json: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.27G [00:00<?, ?B/s]

TensorFlow and JAX classes are deprecated and will be removed in Transformers v5. We recommend migrating to PyTorch classes or pinning your version of Transformers.

TFWav2Vec2Model has backpropagation operations that are NOT supported on CPU. If you wish to train/fine-tune this model, you need a GPU or a TPU
I0000 00:00:1769607397.952069      55 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1769607399.729391      55 service.cc:152] XLA service 0x7757da50 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1769607399.729426      55 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1769607399.729432      55 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1769607400.161657      55 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.
Some weights of the PyTorch model were not used when i

Total transformer layers found: 24
Unfrozen top layers: 6


In [15]:
# latest_ckpt = tf.train.latest_checkpoint(CHECKPOINT_DIR)
# if latest_ckpt:
    # print(f"Resuming from {latest_ckpt}")
model.load_weights("/kaggle/input/checkpoint-ironwall-ai/tensorflow2/default/1/epoch_1.weights.h5")

ckpt_cb = tf.keras.callbacks.ModelCheckpoint(
    filepath=os.path.join(CHECKPOINT_DIR, "epoch_{epoch}.weights.h5"),
    save_weights_only=True,
    save_freq="epoch"
)

/usr/local/lib/python3.12/dist-packages/transformers/generation/tf_utils.py:465: UserWarning: `seed_generator` is deprecated and will be removed in a future version.
  warnings.warn("`seed_generator` is deprecated and will be removed in a future version.", UserWarning)


In [ ]:
for epoch in range(EPOCHS):
    print(f"\n===== Epoch {epoch + 1}/{EPOCHS} =====")

    train_files, train_labels = sample_subset(
        real_train, fake_train, HOURS_PER_EPOCH
    )

    train_gen = AudioGenerator(
        train_files, train_labels,
        batch_size=config.BATCH_SIZE,
        training=True
    )

    model.fit(
        train_gen,
        epochs=1,
        validation_data=val_gen,
        class_weight=CLASS_WEIGHT,
        callbacks=[ckpt_cb],
        verbose=1
    )
model.export("model")
print("Final model saved.")


===== Epoch 1/2 =====
INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).


INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).


INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).


INFO:tensorflow:Reduce to /job:localhost/replica:0/task:0/device:CPU:0 then broadcast to ('/job:localhost/replica:0/task:0/device:CPU:0',).


54000/54000 [==============================] - 13962s 259ms/step - loss: 0.6653 - accuracy: 0.7471 - val_loss: 0.5632 - val_accuracy: 0.7243


/usr/local/lib/python3.12/dist-packages/transformers/generation/tf_utils.py:465: UserWarning: `seed_generator` is deprecated and will be removed in a future version.
  warnings.warn("`seed_generator` is deprecated and will be removed in a future version.", UserWarning)



===== Epoch 2/2 =====
54000/54000 [==============================] - 13961s 259ms/step - loss: 0.6057 - accuracy: 0.7735 - val_loss: 0.5332 - val_accuracy: 0.7505
INFO:tensorflow:Assets written to: secure_wav2vec2_detector/assets


INFO:tensorflow:Assets written to: secure_wav2vec2_detector/assets


Final model saved.
